<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 5 — Your Own Portfolio: Real Data and Business Rules

In Studio 4 you solved Ms. Womack's model and ran it on stocks someone else picked. Tonight it is your turn:

1. **Part 1 · Download your own stocks** from Yahoo Finance and turn the prices into monthly returns.
2. **Part 2 · Ms. Womack's model, plus business rules.** The yes/no decisions from Module 5: if you invest in a
   stock at all, put in at least 2%; never more than 10% in any one stock; hold at least one stock from the
   sectors you choose.
3. **Part 3 · What the rules cost.** The efficient frontier with and without the rules, side by side.

All of the code is written. You choose the stocks and the rules. When you see **Predict first**, write down your
guess before you run the next cell.

## Import modules (download Pyomo, ipopt, Bonmin and yfinance)

Run this first, every time. It installs Pyomo with two solvers: **ipopt** (nonlinear, from Module 4) and
**Bonmin** (nonlinear *and* yes/no variables, new tonight). It also installs `yfinance`. About a minute.

In [ ]:
# install Pyomo, the solvers and yfinance in Colab (idaes-pse brings ipopt and Bonmin)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
from pyomo.environ import *
from IPython.display import display, Math

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# where the setup cell put the solvers
ipopt_path = '/content/bin/ipopt'
bonmin_path = '/content/bin/bonmin'

# Part 1 · Download your own stocks

### Step 1 · Pick your stocks

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio5/themes_infographic.png" width="100%" alt="The Studio 5 themes: 15 ways to pick your stocks"/>

Four ways to get your stocks. Set `my_theme` to one of them:

| `my_theme` | You get | |
|---|---|---|
| `'starter'` | the 20 stocks in `my_stocks` below | edit them freely: one line per stock, ticker and sector |
| a theme, for example `'ai_chips'` | one of 12 ready-made themes (the list is in the cell) | Connecticut companies, oil and gas, travel, dividends... |
| `'draw'` | a theme drawn from your NetID | everyone in the room gets a different list |
| `'dartboard'` | 20 random S&P 500 stocks | a monkey throwing darts: can Ms. Womack still beat the market? |

Keep at least **12 stocks**: tonight's rules need room to spread the money out. Not sure of a ticker? Search the
company name on finance.yahoo.com.

#### 🎯 Your choice: where do your stocks come from?

Set `my_theme` and `my_netid` in the next cell. Everything after it follows from your choice.

In [ ]:
# how do you want to get your stocks?
#   'starter'    the 20 stocks in the next cell (edit them there)
#   a theme:     'connecticut', 'ai_chips', 'oil_and_gas', 'banks_and_payments', 'dividends', 'healthcare',
#                'food_and_restaurants', 'travel', 'defense_and_aerospace', 'utilities_and_power', 'shopping',
#                'sports_and_entertainment'
#   'draw'       a theme drawn from your NetID
#   'dartboard'  20 random S&P 500 stocks
my_theme = 'starter'

# your NetID, for example abc12345 ('draw' and 'dartboard' use it, so your list is yours)
my_netid = ''

The next cell builds your list from your choice. If you picked `'starter'`, edit the stocks right here: one line per
stock, the ticker on the left and its sector on the right. For every other choice, leave this cell as it is.

In [ ]:
# the starter list: ticker -> sector (one line per stock)
my_stocks = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'NVDA': 'Technology',
    'GOOGL': 'Communication Services',
    'META': 'Communication Services',
    'JPM': 'Financials',
    'GS': 'Financials',
    'V': 'Financials',
    'JNJ': 'Health Care',
    'LLY': 'Health Care',
    'UNH': 'Health Care',
    'XOM': 'Energy',
    'CVX': 'Energy',
    'KO': 'Consumer Staples',
    'WMT': 'Consumer Staples',
    'COST': 'Consumer Staples',
    'AMZN': 'Consumer Discretionary',
    'HD': 'Consumer Discretionary',
    'CAT': 'Industrials',
    'LMT': 'Industrials',
}

# twelve themes (every company was in the S&P 500 on Dec 31, 2025)
themes = {
    'connecticut': ['TRV', 'HIG', 'CI', 'SWK', 'RTX', 'OTIS', 'CARR', 'URI', 'SYF', 'HUBB', 'ES', 'IT', 'FDS', 'CHTR', 'XOM', 'PG', 'MSFT', 'UNH'],
    'ai_chips': ['NVDA', 'AMD', 'AVGO', 'MU', 'LRCX', 'AMAT', 'KLAC', 'INTC', 'QCOM', 'TXN', 'ADI', 'ANET', 'MSFT', 'GOOGL', 'META', 'ORCL', 'LLY', 'XOM'],
    'oil_and_gas': ['XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PSX', 'MPC', 'VLO', 'KMI', 'WMB', 'OKE', 'HAL', 'DVN', 'FANG', 'MSFT', 'JNJ'],
    'banks_and_payments': ['JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'SCHW', 'V', 'MA', 'AXP', 'PYPL', 'COF', 'BLK', 'USB', 'AAPL', 'UNH', 'CVX'],
    'dividends': ['KO', 'PEP', 'PG', 'JNJ', 'MRK', 'ABBV', 'VZ', 'T', 'DUK', 'SO', 'XOM', 'CVX', 'MMM', 'IBM', 'MO', 'PM', 'CSCO', 'O'],
    'healthcare': ['LLY', 'JNJ', 'UNH', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'ISRG', 'AMGN', 'GILD', 'CVS', 'MDT', 'BSX', 'SYK', 'AAPL', 'XOM'],
    'food_and_restaurants': ['MCD', 'SBUX', 'CMG', 'YUM', 'DPZ', 'KO', 'PEP', 'MDLZ', 'HSY', 'GIS', 'KHC', 'KDP', 'DRI', 'TSN', 'COST', 'AAPL', 'JNJ', 'XOM'],
    'travel': ['BKNG', 'MAR', 'HLT', 'ABNB', 'EXPE', 'DAL', 'UAL', 'LUV', 'RCL', 'CCL', 'NCLH', 'LVS', 'WYNN', 'MGM', 'UBER', 'AAPL', 'JNJ', 'XOM'],
    'defense_and_aerospace': ['LMT', 'RTX', 'NOC', 'GD', 'BA', 'LHX', 'HII', 'TXT', 'GE', 'HON', 'TDG', 'HWM', 'PLTR', 'LDOS', 'XOM', 'JNJ', 'MSFT'],
    'utilities_and_power': ['NEE', 'DUK', 'SO', 'D', 'AEP', 'EXC', 'SRE', 'XEL', 'PEG', 'ED', 'CEG', 'VST', 'ETR', 'FSLR', 'AES', 'XOM', 'JNJ', 'MSFT'],
    'shopping': ['AMZN', 'WMT', 'COST', 'TGT', 'HD', 'LOW', 'TJX', 'NKE', 'ROST', 'ULTA', 'DG', 'DLTR', 'BBY', 'EBAY', 'AAPL', 'CVS', 'XOM'],
    'sports_and_entertainment': ['DIS', 'NFLX', 'LYV', 'NKE', 'TTWO', 'CMCSA', 'CHTR', 'MAR', 'RCL', 'LVS', 'FOXA', 'HAS', 'GOOGL', 'META', 'AAPL', 'JNJ', 'XOM'],
}

# every S&P 500 company's sector (the official names, with technology called 'Technology' to match the rules)
members = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/sp500_2025-12-31.csv')
sp500_sectors = {}
for i in range(len(members)):
    sector = members['sector'].iloc[i]
    if sector == 'Information Technology':
        sector = 'Technology'
    sp500_sectors[members['ticker'].iloc[i]] = sector

# a number from your NetID, so a draw or a dartboard is always the same for you
seed = 0
for letter in my_netid.strip().lower():
    seed += ord(letter)

if my_theme == 'starter':
    picked = list(my_stocks)
elif my_theme == 'draw':
    if my_netid.strip() == '':
        print("Type your NetID in my_netid first, so the draw is yours.")
    my_theme = sorted(themes)[seed % len(themes)]
    picked = themes[my_theme]
    print("Your NetID drew the theme:", my_theme)
elif my_theme == 'dartboard':
    picked = list(members['ticker'].sample(20, random_state=seed))
    print("The monkey threw 20 darts at the S&P 500.")
elif my_theme in themes:
    picked = themes[my_theme]
else:
    print("Unknown theme:", my_theme, "- using the starter list")
    picked = list(my_stocks)

if my_theme != 'starter':
    my_stocks = {}
    for t in picked:
        my_stocks[t] = sp500_sectors.get(t, 'Other')

# the five years of monthly data we want (first day of the first and last month)
start = '2021-09-01'
end = '2026-10-01'

print(len(my_stocks), "stocks in", len(set(my_stocks.values())), "sectors:", ", ".join(my_stocks))

### Step 2 · Download the prices

This asks Yahoo Finance for the **month-end, adjusted** price of each stock (adjusted for splits and dividends:
remember the NVIDIA split trap from Studio 4.3). It downloads one stock at a time and tells you if any fail.

Yahoo sometimes blocks Colab. If the download fails, the cell falls back to a copy of real prices we saved for
class, so you can keep going either way.

In [ ]:
import yfinance as yf

# collect each stock's prices first, then build the table once (adding columns one at a time is slow)
downloaded = {}
failed = []
for ticker in my_stocks:
    try:
        data = yf.download(ticker, start=start, end=end, interval='1mo', auto_adjust=True, progress=False)
        if len(data) > 12:
            downloaded[ticker] = data['Close'].squeeze()
        else:
            failed.append(ticker)
    except Exception as problem:
        failed.append(ticker)
prices = pd.DataFrame(downloaded)

if len(failed) > 0:
    print("These did not download:", failed)

# the fallback: real month-end prices saved for class (55 stocks, Sep 2021 - Sep 2026)
if len(prices.columns) < 12:
    print("Yahoo did not give us enough stocks, so we are using the saved class copy instead.")
    saved = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/sector_prices_monthly.csv', index_col=0)
    lookup = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/sector_lookup.csv')
    in_copy = []
    for ticker in my_stocks:
        if ticker in saved.columns:
            in_copy.append(ticker)
    prices = saved[in_copy]
    if len(prices.columns) < 12:
        prices = saved[list(lookup['ticker'])]
        my_stocks = {}
        for i in range(len(lookup)):
            my_stocks[lookup.loc[i, 'ticker']] = lookup.loc[i, 'sector']

print(prices.shape[0], "months of prices for", prices.shape[1], "stocks")
prices.tail()

### Step 3 · Prices into monthly returns, then check them

Same three steps as Studio 4.3: returns, then the checks (how many months, anything missing, any impossible moves).

$$r_t = \frac{P_t}{P_{t-1}} - 1$$

In [ ]:
returns = prices.pct_change()
returns = returns.dropna(how='all')

# drop any stock that is missing months (a company that did not exist yet, a ticker that changed)
complete = []
for ticker in returns.columns:
    if returns[ticker].isna().sum() == 0:
        complete.append(ticker)
    else:
        print("dropping", ticker, "- missing", returns[ticker].isna().sum(), "months")
returns = returns[complete]

print("months:", len(returns))
print("stocks:", len(returns.columns))
print("missing values left:", returns.isna().sum().sum())

moves = pd.DataFrame({'worst month': returns.min(), 'best month': returns.max(), 'average month': returns.mean()})
moves.sort_values('worst month').round(3)

#### 📈 Prices and returns, one above the other

The top panel is what \$1 in each of your stocks grew into: prices **trend**, and they all start at \$1 so you can
compare them. The bottom panel is the monthly **return** of an even split of your stocks (green up, red down), with a
gray band from your best to your worst stock that month: returns **jump around zero**. Ms. Womack works with the
bottom panel, because returns are comparable from stock to stock and from month to month, and prices are not.

In [ ]:
fig, (top_panel, bottom_panel) = subplots(2, 1, figsize=(12, 9), sharex=True)

# top: the value of $1 in each stock
growth_of_1 = prices[returns.columns] / prices[returns.columns].iloc[0]
for s in returns.columns:
    top_panel.plot(growth_of_1.index, growth_of_1[s], linewidth=1.3, label=s)
top_panel.axhline(1, color='lightgray', linestyle='--')
top_panel.set_ylabel('value of $1 invested in %s' % prices.index[0].strftime('%b %Y'))
top_panel.set_title('Prices: what $1 in each of your stocks grew into')
top_panel.legend(ncol=6, fontsize=7, loc='upper left')

# bottom: the even split's monthly return, with the best and worst stock each month
average_month = 100 * returns.mean(axis=1)
bar_colors = []
for month_return in average_month:
    if month_return >= 0:
        bar_colors.append('tab:green')
    else:
        bar_colors.append('tab:red')
bottom_panel.fill_between(returns.index, 100 * returns.min(axis=1), 100 * returns.max(axis=1), color='lightgray',
                          alpha=0.6, label='best to worst stock that month')
bottom_panel.bar(returns.index, average_month, width=20, color=bar_colors, label='even split of your stocks')
bottom_panel.axhline(0, color='black', linewidth=0.8)
bottom_panel.set_ylabel('monthly return (%)')
bottom_panel.set_title('Returns: every month, up and down')
bottom_panel.legend(fontsize=8, loc='upper left')
show()

#### 📊 The model's two ingredients: the average month and the ups and downs

Left: every monthly return of every one of your stocks, in one histogram (gray), with the even split on top (blue).
Most months sit near zero; a few are big. Right: each stock's **average month** (the bar) and its **ups and downs**
(the whisker is one standard deviation). The bars are `df_return`, the whiskers come from the diagonal of
`df_cov`: exactly what Ms. Womack trades off.

In [ ]:
fig, (left_panel, right_panel) = subplots(1, 2, figsize=(14, 5))

all_months = 100 * returns.values.flatten()
left_panel.hist(all_months, bins=40, color='lightgray', edgecolor='white', label='every stock, every month')
left_panel.hist(100 * returns.mean(axis=1), bins=20, color='tab:blue', alpha=0.7, label='even split of your stocks')
left_panel.axvline(0, color='black', linewidth=0.8)
left_panel.set_xlabel('monthly return (%)')
left_panel.set_ylabel('number of months')
left_panel.set_title('How big is a typical month?')
left_panel.legend(fontsize=8)

averages = (100 * returns.mean()).sort_values()
spreads = 100 * returns.std()
bar_colors = []
for s in averages.index:
    if averages[s] >= 0:
        bar_colors.append('tab:green')
    else:
        bar_colors.append('tab:red')
right_panel.barh(averages.index, averages.values, xerr=spreads[averages.index].values, color=bar_colors,
                 error_kw={'ecolor': 'gray', 'alpha': 0.6})
right_panel.axvline(0, color='black', linewidth=0.8)
right_panel.set_xlabel('average month (%), whisker = one standard deviation')
right_panel.set_title('Each stock: the average and the ups and downs')
show()

**Look before you model.** Any month worse than -60%? That is usually a split or a data error, not a crash.
Which stock had the best average month? Write it down: we will see whether the model goes all-in on it.

In [ ]:
# save your returns so you can use them again (Files panel on the left, or File > Download)
returns.to_csv('my_monthly_returns.csv')

df_return = returns.mean()
df_cov = returns.cov()
stocks = list(returns.columns)
print("saved my_monthly_returns.csv:", len(stocks), "stocks")

# Part 2 · Ms. Womack's model, plus business rules

### The model you already know (Studio 4.2)

$$
\begin{aligned}
\max \quad & \sum_k \bar r_k \, x_k && \text{(average return)} \\
\text{subject to} \quad & \sum_k x_k = 1 && \text{(invest all of the money)} \\
& \sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j) \le \text{risk limit} && \text{(the risk)} \\
& 0 \le x_k \le 1
\end{aligned}
$$

### New tonight: a yes/no decision for every stock

The business rules are about **whether** you hold a stock, not just how much. So each stock gets a second
decision variable, a **binary** (yes/no) one, exactly like project selection in M5.1:

$$y_k = \begin{cases} 1 & \text{we hold stock } k \\ 0 & \text{we do not} \end{cases}$$

Then each rule is a constraint that **links** $x_k$ (how much) to $y_k$ (whether):

$$
\begin{array}{lcl}
\textbf{Business rule} & \qquad\textbf{Constraint}\qquad & \textbf{What it does} \\[8pt]
\text{Never more than }10\%\text{ in one stock} & x_k \le 0.10\, y_k & \text{if } y_k = 0 \text{, the share must be 0; if } y_k = 1 \text{, up to }10\% \\[10pt]
\text{If you invest at all, at least }2\% & x_k \ge 0.02\, y_k & \text{if } y_k = 1 \text{, at least }2\%\text{ (no tiny }0.3\%\text{ positions)} \\[10pt]
\text{At least one stock from a sector} & \displaystyle\sum_{k \in \text{sector}} y_k \ge 1 & \text{counts the yes/no picks in that sector} \\[14pt]
\text{(optional) At most } K \text{ stocks} & \displaystyle\sum_{k} y_k \le K & \text{caps how many stocks you hold}
\end{array}
$$

Read the first two rows together. Here is every case for one stock:

$$
\begin{array}{c|c|c|l}
y_k & \quad x_k \le 0.10\, y_k \quad & \quad x_k \ge 0.02\, y_k \quad & \text{so } x_k \text{ is...} \\[4pt] \hline
0 & x_k \le 0 & x_k \ge 0 & \text{exactly 0: we do not hold it} \\[4pt]
1 & x_k \le 0.10 & x_k \ge 0.02 & \text{between }2\%\text{ and }10\%
\end{array}
$$

A model with yes/no variables **and** a nonlinear risk is a **mixed-integer nonlinear** program. ipopt cannot handle
the yes/no part, so tonight we use **Bonmin**, the solver from Dave's integer version of Ms. Womack.

### Step 4 · Choose your rules

Edit the dictionary. `None` switches a rule off.

In [ ]:
rules = {
    'min_share': 0.02,                                            # if you invest at all, at least 2%
    'max_share': 0.10,                                            # never more than 10% in one stock
    'must_have_sectors': ['Technology', 'Health Care', 'Energy'],  # at least one stock from each
    'max_stocks': None,                                           # e.g. 12, or None for no limit
}

# more linking rules to play with (all off to start; the rules playground in Part 3 tries them)
extra_rules = {
    'sector_cap': None,     # e.g. 0.30: at most 30% of the money in any one sector
    'min_stocks': None,     # e.g. 12: hold at least 12 stocks
    'must_hold': [],        # e.g. ['AAPL']: these must be held (y = 1)
    'never_hold': [],       # e.g. ['XOM']: these can never be held (y = 0)
    'either_or': [],        # e.g. [['XOM', 'CVX']]: hold one or the other, not both (y_XOM + y_CVX <= 1)
    'if_then': [],          # e.g. [['NVDA', 'AMD']]: if you hold NVDA, you must hold AMD (y_NVDA <= y_AMD)
}

# a quick sanity check: with a cap, you need enough stocks to invest ALL of the money
# (a 10% cap needs at least 10 stocks, because 10 x 10% = 100%)
if rules['max_share'] is not None:
    cap = round(100 * rules['max_share'])
    fewest = int(np.ceil(100 / cap))
    if len(stocks) < fewest:
        print("Not enough stocks: with a %d%% cap you need at least %d stocks to invest all of the money (%d x %d%% = 100%%)."
              % (cap, fewest, fewest, cap))
        print("You have %d. Add stocks, or raise max_share." % len(stocks))
    else:
        print("OK: with a %d%% cap you need at least %d stocks to invest all of the money (%d x %d%% = 100%%). You have %d."
              % (cap, fewest, fewest, cap, len(stocks)))

### Step 5 · The model, with a switch for the rules

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio5/solve_portfolio_flowchart.png" width="100%" alt="What solve_portfolio does, step by step"/>

Read the picture top to bottom, then find each step in the code below: the comments in the code use the same words.

`solve_portfolio(risk_limit, use_rules)` is Studio 4.2's model plus the yes/no variables. With `use_rules=False`
it is exactly Ms. Womack (solved with ipopt); with `use_rules=True` it adds every rule in the dictionary
(solved with Bonmin).

In [ ]:
def solve_portfolio(risk_limit, use_rules):
    model = ConcreteModel()

    # how much of the money goes in each stock
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # the risk: the weighted covariance matrix, summed
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    if use_rules:
        # the yes/no decision: do we hold stock s?
        model.y = Var(stocks, domain=Binary)
        for s in stocks:
            model.y[s] = 1
        for s in stocks:
            if rules['max_share'] is not None:
                model.constraints.add(model.x[s] <= rules['max_share'] * model.y[s])
            else:
                model.constraints.add(model.x[s] <= model.y[s])
            if rules['min_share'] is not None:
                model.constraints.add(model.x[s] >= rules['min_share'] * model.y[s])
        for sector in rules['must_have_sectors']:
            picks = 0
            count = 0
            for s in stocks:
                if my_stocks.get(s) == sector:
                    picks += model.y[s]
                    count += 1
            if count > 0:    # a sector you have no stocks in is skipped
                model.constraints.add(picks >= 1)
        if rules['max_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held <= rules['max_stocks'])
        # the extra linking rules
        if extra_rules['sector_cap'] is not None:
            for sector in set(my_stocks.values()):
                in_sector = 0
                count = 0
                for s in stocks:
                    if my_stocks.get(s) == sector:
                        in_sector += model.x[s]
                        count += 1
                if count > 0:
                    model.constraints.add(in_sector <= extra_rules['sector_cap'])
        if extra_rules['min_stocks'] is not None:
            held = 0
            for s in stocks:
                held += model.y[s]
            model.constraints.add(held >= extra_rules['min_stocks'])
        for s in extra_rules['must_hold']:
            if s in stocks:
                model.constraints.add(model.y[s] == 1)
        for s in extra_rules['never_hold']:
            if s in stocks:
                model.constraints.add(model.y[s] == 0)
        for pair in extra_rules['either_or']:
            if pair[0] in stocks and pair[1] in stocks:
                model.constraints.add(model.y[pair[0]] + model.y[pair[1]] <= 1)
        for pair in extra_rules['if_then']:
            if pair[0] in stocks and pair[1] in stocks:
                model.constraints.add(model.y[pair[0]] <= model.y[pair[1]])
        results = SolverFactory('bonmin', executable=bonmin_path).solve(model)
    else:
        results = SolverFactory('ipopt', executable=ipopt_path).solve(model)

    if results.solver.termination_condition != TerminationCondition.optimal:
        return None
    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    held = {}
    if use_rules:
        for s in stocks:
            held[s] = model.y[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights, 'held': held,
            'risk_limit': risk_limit}


print("solve_portfolio is ready")

### Step 6 · One risk limit, with and without the rules

To compare the two models fairly, we give them **the same risk limit**. Which one? A simple yardstick: **the risk of
an even split**, the same share of the money in every one of your stocks. Both models can always reach it (an even
split keeps every rule), so neither one is set up to fail.

**Where the number comes from.** The risk of any portfolio is the weighted covariance matrix, summed (Studio 4):

$$\text{risk} = \sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j)$$

In an even split of $n$ stocks every share is $x_i = \frac{1}{n}$, so every cell of the covariance matrix gets the
same weight, $\frac{1}{n} \times \frac{1}{n} = \frac{1}{n^2}$:

$$\text{risk of an even split} = \frac{1}{n^2} \sum_i \sum_j \text{Cov}(i,j)$$

That is just the **average of all $n \times n$ cells** of your covariance matrix. Here it is first with only three of
your stocks.

In [ ]:
# a small example: three of your stocks, a third of the money in each
three = stocks[:3]
small_cov = df_cov.loc[three, three]
weight = 1 / 3
cells = pd.DataFrame(index=three, columns=three, dtype=float)
for i in three:
    for j in three:
        cells.loc[i, j] = weight * weight * small_cov.loc[i, j]

print("The covariance matrix of", three, "(monthly):")
display(small_cov.round(5))
print("Each cell times (1/3) x (1/3):")
display(cells.round(6))
print("Add up all 9 cells: the risk of an even split of these 3 stocks = %.5f" % cells.values.sum())

Now the same thing with **all** of your stocks: every cell of your covariance matrix times $\frac{1}{n^2}$, all added
up. The left picture is the three-stock example; the right one is the whole matrix. The total on the right is the
risk limit for Step 6.

**Before you run the next cell:** will the rules make the best return go up, down, or stay the same?

In [ ]:
even = {}
for s in stocks:
    even[s] = 1 / len(stocks)
even_risk = 0
for i in stocks:
    for j in stocks:
        even_risk += even[i] * even[j] * df_cov.loc[i, j]
risk_limit = even_risk
print("risk limit: %.4f (the risk of splitting your money evenly across all %d stocks)" % (risk_limit, len(stocks)))

# the picture: the 3-stock example, and every cell of your matrix times 1/n^2
weighted = df_cov.loc[stocks, stocks] / len(stocks) ** 2
fig, (small_panel, big_panel) = subplots(1, 2, figsize=(14, 5.5))
small_panel.imshow(cells.values.astype(float), cmap='Oranges')
for a in range(3):
    for b in range(3):
        small_panel.text(b, a, "%.5f" % cells.iloc[a, b], ha='center', va='center', fontsize=11)
small_panel.set_xticks(range(3))
small_panel.set_xticklabels(three)
small_panel.set_yticks(range(3))
small_panel.set_yticklabels(three)
small_panel.set_title("3 stocks: (1/3) x (1/3) x Cov(i,j)\nthe 9 cells add up to %.5f" % cells.values.sum())
shown = big_panel.imshow(weighted.values, cmap='Oranges')
big_panel.set_xticks(range(len(stocks)))
big_panel.set_xticklabels(stocks, rotation=90, fontsize=7)
big_panel.set_yticks(range(len(stocks)))
big_panel.set_yticklabels(stocks, fontsize=7)
big_panel.set_title("all %d stocks: (1/%d) x (1/%d) x Cov(i,j)\nthe %d cells add up to %.4f = the risk limit"
                    % (len(stocks), len(stocks), len(stocks), len(stocks) ** 2, risk_limit))
colorbar(shown, ax=big_panel, fraction=0.046)
show()

free = solve_portfolio(risk_limit, use_rules=False)
ruled = solve_portfolio(risk_limit, use_rules=True)

held_free = 0
held_ruled = 0
for s in stocks:
    if free['weights'][s] > 0.001:
        held_free += 1
    if ruled['weights'][s] > 0.001:
        held_ruled += 1
# what each portfolio would earn on $10,000 in an average month (based on the past five years)
invested = 10000
free_dollars = invested * free['return']
ruled_dollars = invested * ruled['return']
cost_points = 100 * (free['return'] - ruled['return'])
cost_dollars = free_dollars - ruled_dollars

print()
print("Same risk limit for both. On $10,000, in an average month:")
print()
print("                     average return    you would earn    stocks held")
free_text = "$%.0f a month" % free_dollars
ruled_text = "$%.0f a month" % ruled_dollars
print("  no rules           %5.2f%% a month    %-16s  %d" % (100 * free['return'], free_text, held_free))
print("  with your rules    %5.2f%% a month    %-16s  %d" % (100 * ruled['return'], ruled_text, held_ruled))
print()
print("Following the rules costs %.2f percentage points a month:" % cost_points)
print("about $%.0f less a month on $10,000, or about $%.0f less a year." % (cost_dollars, 12 * cost_dollars))

#### ⚖️ Same risk, three ways to split the money

The next cell draws three portfolios side by side. All three have **the same risk**, the Step 6 risk limit, so the
only thing that differs is **how the money is split**:

* **Even split** (blue): the same share in every stock. This is the yardstick; it reaches the risk limit without any
  thinking.
* **No rules** (gray): Ms. Womack, free to put the money wherever the past paid best.
* **With your rules** (orange): Ms. Womack inside your floor, your cap and your sectors (the dotted and dashed lines).

Each panel's title shows its average monthly return. Run the cell, then answer:

1. How much more return does Ms. Womack find than the even split, at the same risk?
2. Which stocks does the no-rules portfolio pile into, and what do your rules do to them?

In [ ]:
# three portfolios at the SAME risk limit: the even split (the yardstick), no rules, with your rules
even_return = 0
for s in stocks:
    even_return += even[s] * df_return[s]
portfolios = {
    'even split': (even, even_return, 'tab:blue'),
    'no rules': (free['weights'], free['return'], 'gray'),
    'with your rules': (ruled['weights'], ruled['return'], 'tab:orange'),
}
both = pd.Series(free['weights']) + pd.Series(ruled['weights'])
order = list(both.sort_values(ascending=False).index)
fig, panels = subplots(1, 3, figsize=(16, 5), sharey=True)
for panel, name in zip(panels, portfolios):
    weights, average, color = portfolios[name]
    shares = []
    for s in order:
        shares.append(100 * weights[s])
    panel.barh(order, shares, color=color)
    if name == 'with your rules':
        panel.axvline(100 * rules['max_share'], color='black', linestyle='--', linewidth=1, label='10% cap')
        panel.axvline(100 * rules['min_share'], color='black', linestyle=':', linewidth=1, label='2% floor')
        panel.legend(loc='lower right')
    panel.set_title("%s\nreturn %.2f%% a month" % (name, 100 * average))
    panel.set_xlabel('share of the money (%)')
panels[0].invert_yaxis()
fig.suptitle('Same risk limit (%.4f), three ways to split the money' % risk_limit, fontsize=14, y=1.03)
show()

**Read the bars.** All three portfolios have the same risk. The even split (blue) is the yardstick: it reaches
that risk without thinking. Ms. Womack with no rules (gray) gets much more return from the same risk by piling into
a few stocks with the best past averages. With your rules (orange) every position sits between the dotted 2% floor
and the dashed 10% cap, so the money is spread over at least 10 stocks.
Adding a constraint can never make the best return go up (the rule-free answer was always available without the
rules), so the rules can only cost you return. What they buy you is a portfolio you could actually defend to Ms.
Womack.

# Part 3 · What the rules cost: two frontiers

Solve both models at many risk limits and draw both efficient frontiers. The gap between the curves is the price of
the business rules, at every level of risk. (The rules version takes a little longer: Bonmin tries yes/no
combinations.)

In [ ]:
# risk limits from just above the safest reachable mix to 40% past the riskiest single stock,
# so you can see the no-rules line go flat too
top = df_cov.values.diagonal().max()
levels = np.linspace(even_risk * 0.5, top * 1.4, 18)

free_curve = {}
free_weights = {}
ruled_curve = {}
ruled_results = {}     # every answer Bonmin gave us, kept for the quality check below
for r in levels:
    a = solve_portfolio(r, use_rules=False)
    if a is not None:
        free_curve[r] = a['return']
        free_weights[r] = a['weights']
    b = solve_portfolio(r, use_rules=True)
    if b is not None:
        ruled_curve[r] = b['return']
        ruled_results[r] = b

# where does each line go flat? solve once with the highest risk limit, then read off how much risk it used
free_best = solve_portfolio(levels[-1], use_rules=False)
free_flat_risk = free_best['risk']
best_stock = stocks[0]
for s in stocks:
    if free_best['weights'][s] > free_best['weights'][best_stock]:
        best_stock = s
print("with no rules, the best return is %.2f%% a month: %.0f%% of the money in %s, at a risk of %.4f"
      % (100 * free_best['return'], 100 * free_best['weights'][best_stock], best_stock, free_flat_risk))
print("past that point the gray line is flat: there is nothing better to buy with extra risk")
print()
ruled_best = solve_portfolio(top, use_rules=True)
flat_risk = ruled_best['risk']
print("with your rules, the best return is %.2f%% a month, and it only needs a risk of %.4f"
      % (100 * ruled_best['return'], flat_risk))
print("past that point, allowing more risk buys nothing: your rules are what hold the return back")

# a closer look at the rules line, from the lowest risk limit to just past the flat point
zoom_levels = np.linspace(levels[0], flat_risk * 1.3, 15)
ruled_zoom = {}
ruled_weights = {}
for r in zoom_levels:
    a = solve_portfolio(r, use_rules=False)
    if a is not None:
        free_curve[r] = a['return']
        free_weights[r] = a['weights']
    b = solve_portfolio(r, use_rules=True)
    if b is not None:
        ruled_zoom[r] = b['return']
        ruled_weights[r] = b['weights']
        ruled_results[r] = b


def find_corner(curve):
    # the top-left corner: scale both axes from 0 to 1, draw a straight line from the first point to the last,
    # and take the point that sits farthest above that line. Past it, extra risk buys less and less return.
    risks = list(curve.keys())
    returns = list(curve.values())
    if returns[-1] == returns[0]:
        return risks[0]
    best_risk = risks[0]
    best_gap = -1
    for i in range(len(risks)):
        x = (risks[i] - risks[0]) / (risks[-1] - risks[0])
        y = (returns[i] - returns[0]) / (returns[-1] - returns[0])
        if y - x > best_gap:
            best_gap = y - x
            best_risk = risks[i]
    return best_risk


corner_risk = find_corner(ruled_zoom)
corner_return = ruled_zoom[corner_risk]
print("top-left corner of your orange line: risk limit %.4f, return %.2f%% a month" % (corner_risk, 100 * corner_return))

fig, panels = subplots(1, 2, figsize=(15, 5.5))

left = panels[0]
free_risks = sorted(free_curve.keys())
free_returns = []
for r in free_risks:
    free_returns.append(free_curve[r])
left.plot(free_risks, free_returns, marker='o', color='gray', label='no rules')
# the orange line on the left uses both sets of points, so the zoomed-in corner sits on it
ruled_all = dict(ruled_curve)
ruled_all.update(ruled_zoom)
ruled_risks = sorted(ruled_all.keys())
ruled_returns = []
for r in ruled_risks:
    ruled_returns.append(ruled_all[r])
left.plot(ruled_risks, ruled_returns, marker='o', color='tab:orange', label='with your rules')
left.axvline(flat_risk, color='tab:orange', linestyle=':', label='rules line goes flat')
left.axvline(free_flat_risk, color='gray', linestyle=':', label='no-rules line goes flat (all in %s)' % best_stock)
left.plot(corner_risk, corner_return, marker='*', markersize=18, color='black', linestyle='none', label='top-left corner')
left.set_xlabel('Risk limit (variance of the monthly return)')
left.set_ylabel('Best average monthly return')
left.set_title('Both frontiers: the gap is the price of the rules')
left.legend()

right = panels[1]
right.plot(list(ruled_zoom.keys()), list(ruled_zoom.values()), marker='o', color='tab:orange', label='with your rules')
right.axvline(flat_risk, color='tab:orange', linestyle=':', label='flat from here on')
right.plot(corner_risk, corner_return, marker='*', markersize=18, color='black', linestyle='none', label='top-left corner')
right.set_xlabel('Risk limit (variance of the monthly return)')
right.set_ylabel('Best average monthly return')
right.set_title('Zoomed in: the rules frontier on its own')
right.legend()

show()

**Read the two panels.** On the left, the gray line climbs for a long time: with no rules, more risk lets the model
pile into the few stocks with the best past averages, until **all** of the money sits in your single best stock. From
that point on (the gray dotted line) the gray line is flat too: there is nothing better left to buy with extra risk.
The orange line goes flat almost right away. That is the 10% cap at work.
With at most 10% in any stock, the best you can do is spread the money over the 10 best stocks, and that portfolio
fits under a fairly low risk limit. Past the dotted line, the risk limit is no longer the constraint that binds; your
rules are. The right panel zooms in on the orange line so you can see its own short climb before it flattens.

**The star is the top-left corner**, the spot to pick: the most return for the risk, just before extra risk stops
paying. The code finds it on its own. It scales both axes from 0 to 1, draws a straight line from the first point to
the last, and takes the point farthest above that line. You can always pick a different spot; the Bonus at the end
shows you how to check any risk limit you like.

### Quality control: does Bonmin keep its promises?

Bonmin hands back **numbers**, and numbers can be a little off. A yes/no variable can come back as 0.9999998, or a
stock we said "no" to can keep 0.0000003 of the money, and the solver still calls it optimal because it is inside its
tolerances. So we check every portfolio on both frontiers, at every risk limit, against the rules themselves:

| Check | What must be true |
|---|---|
| clean yes/no | every $y_k$ is 0 or 1 (not 0.98) |
| no means no | if $y_k = 0$ then $x_k = 0$: no money in a stock we said no to |
| yes means a real position | if $y_k = 1$ then $2\% \le x_k \le 10\%$ (no "yes" stock with 0% or 30%) |
| the budget and the risk | the shares add up to 1, and the risk is at or under the limit |
| the sectors and the count | at least one stock held in each must-have sector; no more than `max_stocks` |
| a frontier that never dips | allowing **more** risk can never give a **lower** best return. A dip means the solver stopped before the true best answer |

`tolerance` is how close counts as exact (one ten-thousandth of a percent).

In [ ]:
tolerance = 1e-6


def check_rules(result):
    # every broken promise in one Bonmin answer (an empty list means it passed)
    problems = []
    held_count = 0
    for s in stocks:
        x = result['weights'][s]
        y = result['held'][s]
        if abs(y - round(y)) > tolerance:
            problems.append("%s: y = %.6f is not a clean 0 or 1" % (s, y))
        if round(y) == 0 and x > tolerance:
            problems.append("%s: y = 0 but x = %.6f (money in a stock we said no to)" % (s, x))
        if round(y) == 1:
            held_count += 1
            if rules['min_share'] is not None and x < rules['min_share'] - tolerance:
                problems.append("%s: y = 1 but x = %.4f is under the %.0f%% floor" % (s, x, 100 * rules['min_share']))
            if rules['max_share'] is not None and x > rules['max_share'] + tolerance:
                problems.append("%s: x = %.4f is over the %.0f%% cap" % (s, x, 100 * rules['max_share']))
    total = 0
    for s in stocks:
        total += result['weights'][s]
    # sums over many stocks pick up a little floating-point noise, so the budget and risk checks allow 10x more
    if abs(total - 1) > 10 * tolerance:
        problems.append("the shares add up to %.6f, not 1" % total)
    if result['risk'] > result['risk_limit'] + 10 * tolerance:
        problems.append("the risk %.6f is over the limit %.6f" % (result['risk'], result['risk_limit']))
    for sector in rules['must_have_sectors']:
        have = 0
        count = 0
        for s in stocks:
            if my_stocks.get(s) == sector:
                have += 1
                if round(result['held'][s]) == 1:
                    count += 1
        if have > 0 and count == 0:
            problems.append("no stock held in " + sector)
    if rules['max_stocks'] is not None and held_count > rules['max_stocks']:
        problems.append("%d stocks held, more than %d" % (held_count, rules['max_stocks']))
    if extra_rules['sector_cap'] is not None:
        for sector in set(my_stocks.values()):
            in_sector = 0
            for s in stocks:
                if my_stocks.get(s) == sector:
                    in_sector += result['weights'][s]
            if in_sector > extra_rules['sector_cap'] + tolerance:
                problems.append("%.1f%% in %s, over the sector cap" % (100 * in_sector, sector))
    if extra_rules['min_stocks'] is not None and held_count < extra_rules['min_stocks']:
        problems.append("%d stocks held, fewer than %d" % (held_count, extra_rules['min_stocks']))
    for s in extra_rules['must_hold']:
        if s in stocks and round(result['held'][s]) != 1:
            problems.append(s + " must be held but is not")
    for s in extra_rules['never_hold']:
        if s in stocks and (round(result['held'][s]) != 0 or result['weights'][s] > tolerance):
            problems.append(s + " is held but should never be")
    for pair in extra_rules['either_or']:
        if pair[0] in stocks and pair[1] in stocks:
            if round(result['held'][pair[0]]) + round(result['held'][pair[1]]) > 1:
                problems.append("both %s and %s are held (either-or)" % (pair[0], pair[1]))
    for pair in extra_rules['if_then']:
        if pair[0] in stocks and pair[1] in stocks:
            if round(result['held'][pair[0]]) > round(result['held'][pair[1]]):
                problems.append("%s is held without %s (if-then)" % (pair[0], pair[1]))
    return problems, held_count


rows = []
all_problems = 0
previous_return = None
for r in sorted(ruled_results):
    result = ruled_results[r]
    problems, held_count = check_rules(result)
    smallest = 1
    largest = 0
    worst_y_gap = 0
    for s in stocks:
        if round(result['held'][s]) == 1:
            smallest = min(smallest, result['weights'][s])
            largest = max(largest, result['weights'][s])
        worst_y_gap = max(worst_y_gap, abs(result['held'][s] - round(result['held'][s])))
    if previous_return is not None and result['return'] < previous_return - tolerance:
        problems.append("the best return went DOWN as the risk limit went up (a dip)")
    previous_return = result['return']
    all_problems += len(problems)
    rows.append({'risk limit': round(r, 6), 'return (%)': round(100 * result['return'], 3), 'stocks held': held_count,
                 'smallest held (%)': round(100 * smallest, 2), 'largest (%)': round(100 * largest, 2),
                 'y furthest from 0 or 1': '%.1e' % worst_y_gap, 'check': 'PASS' if len(problems) == 0 else 'FAIL'})
    for problem in problems:
        print("risk limit %.5f: %s" % (r, problem))

qc_table = pd.DataFrame(rows)
if all_problems == 0:
    print("QUALITY CONTROL PASSED: all", len(qc_table), "Bonmin portfolios keep every rule, and the frontier never dips.")
else:
    print("QUALITY CONTROL FOUND", all_problems, "PROBLEMS (listed above). Tell Dr. Dave which risk limits.")
qc_table

**The picture of the promise.** Every dot is one stock at one risk limit: across, its yes/no value $y$; up, its share
$x$. The rules allow only the two green boxes: $y = 0$ with $x = 0$, or $y = 1$ with $x$ between the floor and the cap.
A dot anywhere else is a broken promise.

In [ ]:
dots_y = []
dots_x = []
for r in ruled_results:
    for s in stocks:
        dots_y.append(ruled_results[r]['held'][s])
        dots_x.append(100 * ruled_results[r]['weights'][s])
# a little sideways jitter so the dots do not hide behind each other
spread = np.random.default_rng(1).uniform(-0.04, 0.04, len(dots_y))

figure(figsize=(8, 5))
floor = 0
if rules['min_share'] is not None:
    floor = 100 * rules['min_share']
cap = 100
if rules['max_share'] is not None:
    cap = 100 * rules['max_share']
fill_between([-0.12, 0.12], -0.3, 0.3, color='tab:green', alpha=0.25, label='allowed: y = 0, x = 0')
fill_between([0.88, 1.12], floor, cap, color='tab:green', alpha=0.25, label='allowed: y = 1, floor <= x <= cap')
scatter(np.array(dots_y) + spread, dots_x, s=12, color='black', alpha=0.5)
xticks([0, 1], ['y = 0 (do not hold)', 'y = 1 (hold)'])
xlim(-0.4, 1.4)
ylabel('x, share of the money (%)')
title('%d stocks x %d risk limits: every dot must sit in a green box' % (len(stocks), len(ruled_results)))
legend(loc='upper center')
show()

### The rules playground: sensitivity analysis for your rules

This is **sensitivity analysis**, like the what-if questions you asked of linear programs: change the constraints,
re-solve, watch the objective. In a linear program a shadow price tells you what loosening a constraint is worth.
With yes/no variables shadow prices are no longer reliable, so we do the honest thing instead: **solve again** with
and without each rule.

First a **ladder**: start with no rules, then add one rule at a time, always at the same risk limit (the Step 6
yardstick). Each bar is the best return the model can still find. The drop from one bar to the next is the price of
that one rule.

| Step | Rule | Constraint |
|---|---|---|
| 1 | no rules | (Ms. Womack) |
| 2 | your floor and cap | $f\,y_k \le x_k \le c\,y_k$ |
| 3 | must-have sectors | $\sum_{\text{sector}} y_k \ge 1$ |
| 4 | no sector over 30% | $\sum_{\text{sector}} x_k \le 0.3$ |
| 5 | at least 12 stocks | $\sum_k y_k \ge 12$ |
| 6 | either-or: your two biggest picks | $y_a + y_b \le 1$ |
| 7 | if-then: top pick needs your calmest stock | $y_a \le y_b$ |

Then make it yours: change `ladder_extras` (or `extra_rules` in Step 4) and run it again. Every answer goes through
the quality check from above.

In [ ]:
# the two picks for the either-or and if-then steps come from YOUR no-rules portfolio at the Step 6 risk limit
free_order = pd.Series(free['weights']).sort_values(ascending=False)
top_pick = free_order.index[0]
second_pick = free_order.index[1]
variances = df_cov.values.diagonal()
calmest = stocks[int(np.argmin(variances))]
if calmest == top_pick:
    calmest = stocks[int(np.argsort(variances)[1])]

ladder_extras = [
    ('1. no rules', None),
    ('2. floor + cap', {}),
    ('3. + must-have sectors', {}),
    ('4. + no sector over 30%', {'sector_cap': 0.30}),
    ('5. + at least 12 stocks', {'min_stocks': 12}),
    ('6. + %s or %s, not both' % (top_pick, second_pick), {'either_or': [[top_pick, second_pick]]}),
    ('7. + if %s then %s' % (top_pick, calmest), {'if_then': [[top_pick, calmest]]}),
]

your_rules = dict(rules)
your_extras = dict(extra_rules)


def ladder_step(k):
    # the rules for step k of the ladder: each step keeps every rule before it
    step_rules = dict(your_rules)
    step_extras = {'sector_cap': None, 'min_stocks': None, 'must_hold': [], 'never_hold': [], 'either_or': [], 'if_then': []}
    if k < 2:
        step_rules['must_have_sectors'] = []
    for j in range(1, k + 1):
        changes = ladder_extras[j][1]
        for key in changes:
            step_extras[key] = changes[key]
    return step_rules, step_extras


ladder_rows = []
ladder_checks = 0
for k in range(len(ladder_extras)):
    name = ladder_extras[k][0]
    if ladder_extras[k][1] is None:
        result = solve_portfolio(risk_limit, use_rules=False)
    else:
        rules, extra_rules = ladder_step(k)
        result = solve_portfolio(risk_limit, use_rules=True)
        if result is not None:
            problems, held_count = check_rules(result)
            ladder_checks += len(problems)
    held = 0
    if result is not None:
        for s in stocks:
            if result['weights'][s] > 0.001:
                held += 1
        ladder_rows.append({'step': name, 'return (%)': round(100 * result['return'], 3), 'stocks held': held})
    else:
        ladder_rows.append({'step': name, 'return (%)': np.nan, 'stocks held': 0})
rules = your_rules
extra_rules = your_extras

ladder_table = pd.DataFrame(ladder_rows).set_index('step')
ladder_table['cost of this rule (points)'] = (-ladder_table['return (%)'].diff()).round(3) + 0.0
print("quality check on the ladder:", "every step keeps every rule" if ladder_checks == 0 else "%d problems" % ladder_checks)
print("(a step with no bar means the rules cannot all be met at this risk limit)")

figure(figsize=(11, 4.5))
colors = ['gray'] + ['tab:orange'] * (len(ladder_table) - 1)
bar(range(len(ladder_table)), ladder_table['return (%)'], color=colors)
xticks(range(len(ladder_table)), ladder_table.index, rotation=20, ha='right')
ylabel('Best average return (% a month)')
title('The rules ladder at risk limit %.4f: each step adds one rule' % risk_limit)
low = ladder_table['return (%)'].min()
ylim(max(0, low - 0.3), ladder_table['return (%)'].max() + 0.1)
show()
ladder_table

**Read the ladder.** Which rule cost the most? Some rules cost nothing at all: the model was already obeying them,
so the constraint is not binding. Try your own ladder: change a step in `ladder_extras`, for example
`('6. + never hold TSLA', {'never_hold': ['TSLA']})`, and run the cell again. In Part 4 you will see the twist:
what each rung of the ladder did in 2026.

#### 🎚️ One rule, many settings: what is the cap worth?

The classic sensitivity analysis: keep everything else fixed, move **one** number, re-solve each time. Here the
number is your cap (`max_share`), from 6% to 50%, at the Step 6 risk limit. Where does loosening the cap stop paying?
The dashed line is no rules at all; the star is your own cap.

In [ ]:
cap_levels = [0.06, 0.08, 0.10, 0.12, 0.15, 0.20, 0.30, 0.50]
no_extras = {'sector_cap': None, 'min_stocks': None, 'must_hold': [], 'never_hold': [], 'either_or': [], 'if_then': []}

cap_percent = []
cap_returns = []
for cap_level in cap_levels:
    cap_percent.append(100 * cap_level)
    rules = dict(your_rules)
    rules['max_share'] = cap_level
    extra_rules = no_extras
    result = solve_portfolio(risk_limit, use_rules=True)
    if result is None:
        cap_returns.append(np.nan)
    else:
        cap_returns.append(100 * result['return'])
rules = your_rules
extra_rules = your_extras

no_rules_return = 100 * solve_portfolio(risk_limit, use_rules=False)['return']
figure(figsize=(10, 4.5))
plot(cap_percent, cap_returns, marker='o', color='tab:orange', label='with your floor and sectors')
axhline(no_rules_return, color='gray', linestyle='--', label='no rules at all')
if your_rules['max_share'] in cap_levels:
    star_return = cap_returns[cap_levels.index(your_rules['max_share'])]
    plot(100 * your_rules['max_share'], star_return, marker='*', markersize=18, color='black', linestyle='none',
         label='your cap')
xlabel('cap: the most in any one stock (%)')
ylabel('best average return (% a month)')
title('Sensitivity of the best return to the cap, at risk limit %.4f' % risk_limit)
legend()
show()
sensitivity = pd.DataFrame({'cap (%)': cap_percent, 'best return (% a month)': cap_returns})
sensitivity.round(3)

**Read the curve.** A steep part means the cap is binding and loosening it buys real return. A flat part means the
cap no longer matters: something else (the risk limit, or another rule) is holding you back. A missing point means the
cap is too tight for your number of stocks (with a 6% cap you need at least 17).

### The allocation behind each point on the frontier

Every dot on a frontier is a whole portfolio. This picture stacks the shares, so you can see how the money moves from
stock to stock as you allow more risk. On the no-rules side, watch the right edge: one color takes over the whole
column. That is the gray plateau, everything in one stock. Read it left to right: each color is one stock, and its height is that
stock's share of the money at that risk limit.

In [ ]:
def allocation_table(weights_by_risk):
    # one row per risk limit, one column per stock that ever gets money
    table = pd.DataFrame(weights_by_risk).T.sort_index()
    keep = []
    for s in table.columns:
        if table[s].max() > 0.001:
            keep.append(s)
    table = table[keep].clip(lower=0)
    return table


free_alloc = allocation_table(free_weights)
ruled_alloc = allocation_table(ruled_weights)

# the same color for a stock in both panels
all_held = list(free_alloc.columns)
for s in ruled_alloc.columns:
    if s not in all_held:
        all_held.append(s)
colors = {}
for k in range(len(all_held)):
    colors[all_held[k]] = cm.tab20(k % 20)

fig, panels = subplots(1, 2, figsize=(15, 5.5), sharey=True)

free_colors = []
for s in free_alloc.columns:
    free_colors.append(colors[s])
panels[0].stackplot(free_alloc.index, free_alloc.T.values, colors=free_colors)
panels[0].set_title('No rules')
panels[0].set_xlabel('Risk limit (variance of the monthly return)')
panels[0].set_ylabel('Share of the money')

ruled_colors = []
for s in ruled_alloc.columns:
    ruled_colors.append(colors[s])
panels[1].stackplot(ruled_alloc.index, ruled_alloc.T.values, colors=ruled_colors)
panels[1].axvline(corner_risk, color='black', linestyle='--', linewidth=2)
panels[1].set_title('With your rules (zoomed range; dashed line = top-left corner)')
panels[1].set_xlabel('Risk limit (variance of the monthly return)')

handles = []
for s in all_held:
    handles.append(Rectangle((0, 0), 1, 1, color=colors[s]))
panels[1].legend(handles, all_held, loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8)
show()

print("your portfolio at the top-left corner (percent of the money):")
corner_mix = ruled_alloc.loc[corner_risk]
corner_mix = corner_mix[corner_mix > 0.001].sort_values(ascending=False)
print((100 * corner_mix).round(1).to_string())

**Read the allocation plots.** With no rules, the money collapses into just a few stocks as you allow more risk:
the bands narrow until one or two colors fill the top. With your rules no band can grow past 10% or shrink below 2%,
so the stack stays spread out and, past the flat point, stops changing at all.

# Part 4 · Backtest: would your fund have beaten the S&P 500 in 2026?

Everything so far used all five years (Oct 2021 to Sep 2026) to pick the portfolio. That is like grading a test with
the answer key in hand: the model has already seen every month it would be judged on. A fair test **hides** some
months from the model.

So pretend it is **New Year's Eve, Dec 31, 2025.** You know 2021 to 2025, and nothing about 2026.

* **4a · Split the data** into the fit years (Oct 2021 to Dec 2025) and the test months (Jan to Sep 2026).
* **4b · Fit on the fit years only.** Same models, same rules, same top-left corner method as Part 3.
* **4c · Buy on Dec 31, 2025 and hold through 2026.** \$10,000 in each portfolio, plus two baselines: an even split
  of your stocks, and the S&P 500 (the SPY fund).
* **4d · The scorecard.** Did your fund beat the S&P 500? Did the model deliver what it promised?

### 4a · Split the data

In [ ]:
backtest = {
    'fit_until': '2025-12-31',   # the model may use these months...
    'test_from': '2026-01-01',   # ...and is judged on these, which it never sees
}
fit_years = returns[returns.index <= backtest['fit_until']]
test = returns[returns.index >= backtest['test_from']]
print("fit on: ", fit_years.index[0].strftime('%b %Y'), "to", fit_years.index[-1].strftime('%b %Y'), "(%d months)" % len(fit_years))
print("test on:", test.index[0].strftime('%b %Y'), "to", test.index[-1].strftime('%b %Y'), "(%d months the model never sees)" % len(test))

figure(figsize=(12, 1.8))
barh([0], [len(fit_years)], color='tab:blue')
barh([0], [len(test)], left=[len(fit_years)], color='tab:red')
text(len(fit_years) / 2, 0, 'fit years: %s to %s' % (fit_years.index[0].strftime('%b %Y'), fit_years.index[-1].strftime('%b %Y')),
     ha='center', va='center', color='white', fontsize=11)
text(len(fit_years) + len(test) / 2, 0, '2026: hidden', ha='center', va='center', color='white', fontsize=11)
yticks([])
xlabel('months')
title('The model learns from the blue months and is judged on the red ones')
show()

### 4b · Fit on the fit years only

`solve_portfolio` always reads two variables: `df_return` (each stock's average month) and `df_cov` (the covariance
matrix). Right now both are built from all five years. For a fair test we:

1. save the five-year versions,
2. rebuild `df_return` and `df_cov` from the fit years only (hand the model an older newspaper),
3. solve exactly as in Part 3: the rules frontier, its top-left corner, and both portfolios at that risk limit,
4. put the five-year versions back, so everything after this cell is unchanged.

The risk limit comes out a little different from Part 3's, because the model sees fewer months. The model's
objective value is now its **promise**: the average month it expects, judged by 2021 to 2025.

In [ ]:
# 1. save the five-year versions
five_year_return = df_return
five_year_cov = df_cov

# 2. hand the model the fit years only
df_return = fit_years.mean()
df_cov = fit_years.cov()

# 3. the rules frontier on the fit years, its top-left corner, and both portfolios there
fit_even_risk = 0
for i in stocks:
    for j in stocks:
        fit_even_risk += df_cov.loc[i, j] / len(stocks) ** 2
fit_best = solve_portfolio(df_cov.values.diagonal().max(), use_rules=True)
fit_zoom = {}
for r in np.linspace(fit_even_risk * 0.5, fit_best['risk'] * 1.3, 12):
    b = solve_portfolio(r, use_rules=True)
    if b is not None:
        fit_zoom[r] = b['return']
backtest_limit = find_corner(fit_zoom)
backtest_free = solve_portfolio(backtest_limit, use_rules=False)
backtest_ruled = solve_portfolio(backtest_limit, use_rules=True)

# 4. put the five years back for everything after this cell
df_return = five_year_return
df_cov = five_year_cov

print("top-left corner of the fit-years frontier: risk limit %.4f" % backtest_limit)
print("(Part 3 found %.4f with all five years: a little different, because the model saw fewer months)" % corner_risk)
print()
print("What the model PROMISES, judged by 2021 to 2025:")
print("  no rules         %.2f%% a month" % (100 * backtest_free['return']))
print("  with your rules  %.2f%% a month" % (100 * backtest_ruled['return']))

figure(figsize=(8, 4))
plot(list(fit_zoom.keys()), list(fit_zoom.values()), marker='o', color='tab:orange', label='with your rules (fit years)')
plot(backtest_limit, fit_zoom[backtest_limit], marker='*', markersize=18, color='black', linestyle='none',
     label='top-left corner')
xlabel('Risk limit (variance of the monthly return)')
ylabel('Best average monthly return')
title('The frontier as it looked on Dec 31, 2025')
legend()
show()

bought = pd.DataFrame({'no rules (%)': pd.Series(backtest_free['weights']) * 100,
                       'with your rules (%)': pd.Series(backtest_ruled['weights']) * 100})
bought = bought[(bought['no rules (%)'] > 0.1) | (bought['with your rules (%)'] > 0.1)]
print("What each portfolio buys on Dec 31, 2025:")
bought.sort_values('with your rules (%)', ascending=False).round(1)

### 4c · Buy on Dec 31, 2025, and hold through 2026

**Hold** means we never trade. Each stock's dollars simply grow or shrink with its own monthly returns:

$$\text{value of stock } k \text{ after month } t = \$10{,}000 \times x_k \times (1 + r_{k,\text{Jan}})(1 + r_{k,\text{Feb}}) \cdots (1 + r_{k,t})$$

and the portfolio is worth the sum over its stocks. The even split puts the same share in every stock. The S&P 500
is the SPY fund: what you would have had by just buying the market.

#### 💡 Why multiply, not add? (`cumprod`, in five months)

Say your stock does this for five months: **+10%, −5%, +20%, −10%, +5%**. Start with \$10,000:

| Month | Return | Multiply by $(1 + r)$ | Your money |
|---|---|---|---|
| start | | | \$10,000 |
| 1 | +10% | × 1.10 | \$11,000 |
| 2 | −5% | × 0.95 | \$10,450 |
| 3 | +20% | × 1.20 | \$12,540 |
| 4 | −10% | × 0.90 | \$11,286 |
| 5 | +5% | × 1.05 | \$11,850 |

Each month's return applies to **what you have now**, not to the \$10,000 you started with. A 5% loss after a good run
costs more dollars than a 5% loss on day one. So the returns multiply:

$$\$10{,}000 \times 1.10 \times 0.95 \times 1.20 \times 0.90 \times 1.05 = \$11{,}850$$

Adding them instead (+10 − 5 + 20 − 10 + 5 = +20%) would say \$12,000, which is wrong. That is what `cumprod` does:
a running product of $(1 + r)$, month by month. The next cell shows both.

In [ ]:
five_months = pd.Series([0.10, -0.05, 0.20, -0.10, 0.05], index=['month 1', 'month 2', 'month 3', 'month 4', 'month 5'])
check = pd.DataFrame({
    'return': five_months,
    'multiply (right): cumprod': 10000 * (1 + five_months).cumprod(),
    'add (wrong): cumsum': 10000 * (1 + five_months.cumsum()),
})
check.round(0)

**Now the real thing.** The next cell does the same multiplying for every stock in all three portfolios, plus the
S&P 500: \$10,000 each on Dec 31, 2025, held through 2026. It draws the race and prints the value at the end of
every month.

In [ ]:
start_money = 10000
growth = (1 + test).cumprod()


def hold(weights):
    value = 0
    for s in stocks:
        value = value + start_money * weights[s] * growth[s]
    # the starting point: the Dec 31, 2025 close, before any test month
    start_point = pd.Series([start_money], index=[fit_years.index[-1]])
    return pd.concat([start_point, value])


equal_weights = {}
for s in stocks:
    equal_weights[s] = 1 / len(stocks)

paths = {}
paths['no rules'] = hold(backtest_free['weights'])
paths['with your rules'] = hold(backtest_ruled['weights'])
paths['equal split of your stocks'] = hold(equal_weights)

# the S&P 500, if Yahoo gives it to us
try:
    spy = yf.download('SPY', start=start, end=end, interval='1mo', auto_adjust=True, progress=False)['Close'].squeeze()
    spy_returns = spy.pct_change().reindex(test.index)
    if spy_returns.isna().sum() == 0:
        spy_path = start_money * (1 + spy_returns).cumprod()
        paths['S&P 500 (SPY)'] = pd.concat([pd.Series([start_money], index=[fit_years.index[-1]]), spy_path])
    else:
        print("SPY months did not line up, so the S&P 500 line is left out")
except Exception as problem:
    print("SPY did not download, so the S&P 500 line is left out")

line_colors = {'no rules': 'gray', 'with your rules': 'tab:orange', 'equal split of your stocks': 'tab:blue',
               'S&P 500 (SPY)': 'black'}
figure(figsize=(11, 5))
for name in paths:
    plot(paths[name].index, paths[name].values, marker='o', color=line_colors[name], label=name)
axhline(start_money, color='lightgray', linestyle='--')
ylabel('Value of $10,000')
title('2026, the months the model never saw: $10,000 in each portfolio on Dec 31, 2025')
legend()
show()

# month by month: what each $10,000 was worth at the end of each month
month_by_month = pd.DataFrame(paths).round(0)
labels = ['Dec 31, 2025 (buy)']
for day in month_by_month.index[1:]:
    labels.append('end of ' + day.strftime('%b %Y'))
month_by_month.index = labels
month_by_month

### 4d · The scorecard: did it work?

You bought on Dec 31, 2025 and held through the 2026 months the model never saw. The scorecard answers two
questions, one row per portfolio:

1. **Did your fund beat the S&P 500?** Compare the ending values. The lines printed above the table say it in words.
2. **Did the model deliver what it promised?** Compare the *promised* average month (what the model expected from
   2021 to 2025) with the average month it *actually* earned in 2026.

A third column, worst drop, tells you how bumpy the ride was along the way.

**Every column on one small example.** A fund starts with \$10,000 and goes **+10%, −10%, +5%** over three months:
\$10,000 → \$11,000 → \$9,900 → \$10,395. Its model had promised 2.4% a month.

| Column | What it means | The example |
|---|---|---|
| **ending value (\$)** | what the \$10,000 was worth at the end of the last month | \$10,395 |
| **total return (%)** | how much it grew overall: ending value ÷ 10,000 − 1 | 10,395 ÷ 10,000 − 1 = **3.95%** |
| **average month (%)** | the average of its monthly returns: what **actually** happened | (10 − 10 + 5) ÷ 3 = **1.67%** |
| **promised average month (%)** | the model's objective value from 4b: what it **expected** | **2.4%** (it fell short) |
| **worst drop (%)** | the biggest fall from a high point (the worst stretch) | \$11,000 down to \$9,900 = **10%** |

Two things the example shows:

* **Total return is not average month × months.** 3 × 1.67% = 5%, but the fund only grew 3.95%. Months compound:
  the −10% hit \$11,000, not \$10,000, so it cost more than the +10% had earned (the five-month example in 4c, again).
* **The promise is usually higher than what is delivered.** The model picks the stocks whose past looks best, and
  some of that past was luck. The gap between the two columns is a rough measure of how much.

In [ ]:
promised = {'no rules': backtest_free['return'], 'with your rules': backtest_ruled['return']}
rows = {}
for name in paths:
    path = paths[name]
    monthly = path.pct_change().dropna()
    worst_drop = (1 - path / path.cummax()).max()
    row = {}
    row['ending value ($)'] = round(path.iloc[-1])
    row['total return (%)'] = round(100 * (path.iloc[-1] / start_money - 1), 1)
    row['average month (%)'] = round(100 * monthly.mean(), 2)
    row['promised average month (%)'] = '-'
    if name in promised:
        row['promised average month (%)'] = round(100 * promised[name], 2)
    row['worst drop (%)'] = round(100 * worst_drop, 1)
    rows[name] = row
backtest_table = pd.DataFrame(rows).T

if 'S&P 500 (SPY)' in paths:
    market = paths['S&P 500 (SPY)'].iloc[-1]
    print("The S&P 500 turned $10,000 into $%.0f over the same months." % market)
    for name in ['with your rules', 'no rules', 'equal split of your stocks']:
        gap = paths[name].iloc[-1] - market
        if gap >= 0:
            print("  %-27s beat the S&P 500 by $%.0f" % (name, gap))
        else:
            print("  %-27s trailed the S&P 500 by $%.0f" % (name, -gap))
print()
for name in promised:
    actual = paths[name].pct_change().dropna().mean()
    print("  %-16s promised %.2f%% a month, delivered %.2f%%" % (name, 100 * promised[name], 100 * actual))
backtest_table

**Read the backtest.** First the headline: did your fund (with your rules) beat the S&P 500 in 2026? Then compare
it with its no-rules twin: did the rules cost you return, protect you from a bad drop (the worst drop column), or
both? And look at promised versus delivered: the gap is how much of the past was luck. Nine months is a small sample,
so treat this as a first look, not a verdict. The live leaderboard is the next test.

### 4e · Could a monkey with darts do better?

A classic test: throw darts at the S&P 500, put an equal share in whatever the darts hit, and hold. Below are **200
monkeys**, each with as many random S&P 500 stocks as you have, all bought on Dec 31, 2025 and held through the same
2026 months. Where does your fund land?

In [ ]:
saved = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio6/data/daily_prices.csv', index_col=0, parse_dates=True)
last_test_day = test.index[-1] + pd.offsets.MonthEnd(0)
start_prices = saved.loc[:'2025-12-31'].iloc[-1]
end_prices = saved.loc[:last_test_day].iloc[-1]
growth_2026 = (end_prices / start_prices).dropna()
if 'SPY' in growth_2026.index:
    growth_2026 = growth_2026.drop('SPY')

monkeys = []
for k in range(200):
    darts = growth_2026.sample(len(stocks), random_state=k)
    monkeys.append(start_money * darts.mean())
monkeys = np.array(monkeys)

yours = paths['with your rules'].iloc[-1]
twin = paths['no rules'].iloc[-1]
beaten = 100 * (monkeys < yours).mean()
period = 'Dec 31, 2025 to the end of %s' % test.index[-1].strftime('%b %Y')

figure(figsize=(11, 5))
counts, edges, bars = hist(monkeys, bins=30, color='lightgray', edgecolor='white',
                           label='200 monkeys (%d random stocks, equal split)' % len(stocks))
top = counts.max()
lines = [('your fund', yours, 'tab:orange', '-', 3), ('its no-rules twin', twin, 'gray', '--', 2)]
if 'S&P 500 (SPY)' in paths:
    market = paths['S&P 500 (SPY)'].iloc[-1]
    lines.append(('S&P 500', market, 'black', '-', 2.5))
height = 1.0
for name, money, color, style, width in lines:
    axvline(money, color=color, linestyle=style, linewidth=width, label='%s: $%s' % (name, '{:,.0f}'.format(money)))
    text(money, top * height, ' %s\n $%s' % (name, '{:,.0f}'.format(money)), color=color, fontsize=9, va='top',
         bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85})
    height -= 0.18
xlabel('Value of $10,000 at the end of %s' % test.index[-1].strftime('%b %Y'))
title('Your fund, the S&P 500 and 200 monkeys\nsame months for everyone: %s' % period)
legend(fontsize=8, loc='upper right')
show()

print("Same months for everyone:", period)
if 'S&P 500 (SPY)' in paths:
    gap = yours - market
    if gap >= 0:
        print("  your fund:        $%s, beat the S&P 500 by $%s" % ('{:,.0f}'.format(yours), '{:,.0f}'.format(gap)))
    else:
        print("  your fund:        $%s, trailed the S&P 500 by $%s" % ('{:,.0f}'.format(yours), '{:,.0f}'.format(-gap)))
    print("  its no-rules twin: $%s" % '{:,.0f}'.format(twin))
    print("  the S&P 500:      $%s" % '{:,.0f}'.format(market))
    print("  %.0f%% of the 200 monkeys beat the S&P 500." % (100 * (monkeys > market).mean()))
print("Your fund beat %.0f%% of the monkeys. The middle monkey ended with $%s." % (beaten, '{:,.0f}'.format(np.median(monkeys))))

### 4f · The rules ladder in 2026

The twist. The ladder in Part 3 showed what each rule **costs on paper**. Here each rung is fit on the fit years
only, bought on Dec 31, 2025 and held through 2026, the months the model never saw. A rule that costs return on
paper can still **help** in a year the model never saw, because it stops the model from betting everything on the
past.

In [ ]:
# fit every rung on the fit years only (the same older newspaper as 4b), then hold it through 2026
five_year_return = df_return
five_year_cov = df_cov
df_return = fit_years.mean()
df_cov = fit_years.cov()

rung_rows = []
for k in range(len(ladder_extras)):
    name = ladder_extras[k][0]
    if ladder_extras[k][1] is None:
        result = solve_portfolio(backtest_limit, use_rules=False)
    else:
        rules, extra_rules = ladder_step(k)
        result = solve_portfolio(backtest_limit, use_rules=True)
    if result is None:
        rung_rows.append({'step': name, 'promised (% a month)': np.nan, '$10,000 became': np.nan})
        continue
    held_path = hold(result['weights'])
    rung_rows.append({'step': name, 'promised (% a month)': round(100 * result['return'], 2),
                      '$10,000 became': round(held_path.iloc[-1])})
rules = your_rules
extra_rules = your_extras
df_return = five_year_return
df_cov = five_year_cov

rung_table = pd.DataFrame(rung_rows).set_index('step')
figure(figsize=(11, 4.5))
bar(range(len(rung_table)), rung_table['$10,000 became'], color=['gray'] + ['tab:orange'] * (len(rung_table) - 1))
if 'S&P 500 (SPY)' in paths:
    axhline(paths['S&P 500 (SPY)'].iloc[-1], color='black', linewidth=2, label='S&P 500 (SPY)')
    legend()
xticks(range(len(rung_table)), rung_table.index, rotation=20, ha='right')
ylim(min(9000, rung_table['$10,000 became'].min() - 300), rung_table['$10,000 became'].max() + 300)
ylabel('Value of $10,000 at the end of the test')
title('Every rung of the ladder, held through 2026')
show()
rung_table

**Read the ladder in 2026.** Compare this with the ladder in Part 3. Did the rules that cost the most on paper
also cost the most in 2026? Did any rung beat the S&P 500, or most of the monkeys?

# Part 5 · Put your fund on the class leaderboard

Your fund goes live tonight. This sends the class leaderboard **both** portfolios at your chosen spot on the
frontier (the top-left corner from Part 3, unless you pick another): the one **with your rules** (your fund) and its
**no-rules twin**. Both start with \$10,000 at the next market close, and the leaderboard tracks them every day.

**Who sees what.** The public page shows only your nickname, your theme and the results. Dr. Dave also sees your
NetID and the Google account you sign in with, so he knows whose fund is whose.

**Changing your fund later.** Re-run Steps 4 to 6, Part 3 and this part any time (for example, when a new month of
prices comes in). Each re-run is saved as a new entry with its time and your reason. It counts from the next market
close and starts from what your fund was worth that day. Nothing is ever erased, so your change history is part of
your final report.

Fill in the five lines, then run the next two cells. Dr. Dave writes the class code on the board.

In [ ]:
netid = ""            # your UConn NetID, for example abc12345
nickname = ""         # what the class sees on the leaderboard, for example "Tech Titans"
theme = ""            # one line: what your fund is about, for example "companies I buy from every week"
reason = "first launch"   # when you change your fund later, say why
class_code = ""       # Dr. Dave writes it on the board

# the spot on your orange line: the top-left corner from Part 3, unless you type your own risk limit
entry_limit = corner_risk
# entry_limit = 0.0020

**Send it.** The next cell, in order: works out both portfolios at your risk limit, runs the quality check on the
one with your rules, prints exactly what you are sending, signs you in with Google (a pop-up: pick your account and
click Allow), and prints the receipt the leaderboard saved. If it says "Not sent yet," fill in the lines above and run
it again.

In [ ]:
import json
import sys
import requests

SUBMIT_URL = 'https://us-central1-opim5641-portfolio-f26.cloudfunctions.net/submit-portfolio'
BOARD_URL = 'https://storage.googleapis.com/opim5641-leaderboard-f26/index.html'


def shares(result):
    # the stocks the model actually holds, and their shares of the money
    kept = {}
    for s in stocks:
        if result['weights'][s] > 0.001:
            kept[s] = round(float(result['weights'][s]), 6)
    return kept


entry_free = solve_portfolio(entry_limit, use_rules=False)
entry_ruled = solve_portfolio(entry_limit, use_rules=True)
entry_problems, entry_held = check_rules(entry_ruled)
if len(entry_problems) == 0:
    print("quality control: the portfolio with your rules keeps every rule (%d stocks held)" % entry_held)
else:
    print("quality control found problems in the portfolio with your rules:", entry_problems)

sectors = {}
for s in stocks:
    sectors[s] = my_stocks.get(s, '')

# the Part 4 backtest lines (value of $10,000 each month of the hidden year), so the leaderboard can draw them
backtest_paths = {}
for name in paths:
    one_line = {}
    for day in paths[name].index:
        one_line[str(day.date())] = round(float(paths[name][day]), 2)
    backtest_paths[name] = one_line

entry = {
    'netid': netid.strip().lower(),
    'nickname': nickname.strip(),
    'theme': theme.strip(),
    'reason': reason.strip(),
    'class_code': class_code.strip(),
    'portfolios': {'no_rules': shares(entry_free), 'with_rules': shares(entry_ruled)},
    'promised_monthly_return': {'no_rules': float(entry_free['return']), 'with_rules': float(entry_ruled['return'])},
    'risk_limit': float(entry_limit),
    'rules': rules,
    'extra_rules': extra_rules,
    'stock_theme': my_theme,
    'sectors': sectors,
    'data_window': [str(returns.index[0].date()), str(returns.index[-1].date())],
    'backtest': json.loads(backtest_table.to_json(orient='index')),
    'backtest_paths': backtest_paths,
    'backtest_title': "Backtest: fit on Oct 2021 to Dec 2025, $10,000 held through 2026 (never seen)",
    'backtest_note': "From the Studio 5 notebook (Part 4). The model was fit on Oct 2021 to Dec 2025 only; each "
                     "portfolio was bought on Dec 31, 2025 and held through 2026, months the model never saw.",
    'notebook': 'studio5',
}

side_by_side = pd.DataFrame({'no rules (%)': pd.Series(entry['portfolios']['no_rules']) * 100,
                             'with your rules (%)': pd.Series(entry['portfolios']['with_rules']) * 100}).fillna(0)
print("What you are sending, at risk limit %.4f (share of the money in each stock):" % entry_limit)
print(side_by_side.sort_values('with your rules (%)', ascending=False).round(1).to_string())
print()

missing = []
for field in ['netid', 'nickname', 'theme', 'class_code']:
    if entry[field] == '':
        missing.append(field)

if len(missing) > 0:
    print("Not sent yet. Fill in:", ", ".join(missing), "(in the cell above), then run this cell again.")
elif 'google.colab' not in sys.modules:
    print("Not sent: run this notebook in Google Colab to sign in and submit.")
else:
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    answer = requests.post(SUBMIT_URL, data=json.dumps(entry, default=str), timeout=60,
                           headers={'Authorization': 'Bearer ' + credentials.token, 'Content-Type': 'application/json'})
    receipt = answer.json()
    if receipt.get('ok'):
        received = pd.Timestamp(receipt['received_utc']).tz_convert('America/New_York')
        print("Your fund is in! This is what the leaderboard SAVED. Check that it is the model you meant to send.")
        print("  nickname:     ", receipt['nickname'])
        print("  NetID:        ", receipt['netid'], " (signed in as", receipt['google_email'] + ")")
        print("  theme:        ", receipt.get('theme', ''))
        print("  reason:       ", receipt.get('reason', ''))
        print("  received:     ", received.strftime('%a %b %d, %I:%M %p'), "Eastern")
        if receipt.get('risk_limit') is not None:
            print("  risk limit:   ", "%.4f" % receipt['risk_limit'])
        print("  stocks held:  ", receipt['holdings']['with_rules'], "with your rules,", receipt['holdings']['no_rules'], "with no rules")
        if 'portfolios' in receipt:
            saved = pd.DataFrame({'no rules (%)': pd.Series(receipt['portfolios']['no_rules']) * 100,
                                  'with your rules (%)': pd.Series(receipt['portfolios']['with_rules']) * 100}).fillna(0)
            print()
            print(saved.sort_values('with your rules (%)', ascending=False).round(1).to_string())
            print()
        print("  it counts from the next market close (4 PM Eastern on a weekday)")
        if receipt.get('fund_page'):
            print("  your fund's page:", receipt['fund_page'], "(ready within 5 minutes)")
        if receipt.get('record'):
            print("  your private record:", receipt['record'])
            print("     (your final-project dashboard reads this link: save it, and keep it to yourself)")
        print("  the leaderboard: ", BOARD_URL, "(updates every 5 minutes)")
    else:
        print("Not accepted:", receipt.get('problem'))

### 🤖 Prompts to try with Claude

Claude is a good partner for this notebook. Paste the prompt **together with your own code or numbers**, then check
what it tells you against the notebook's own output. A few to start with:

**Picking stocks**
* "Suggest 20 S&P 500 stocks for a fund about *[something you care about]*, with at least one Technology, one Health
  Care and one Energy stock. Give them to me as a Python dictionary, ticker to sector, like my `my_stocks`."
* "Here is my `my_stocks`. Which of these companies are really betting on the same thing? Which two would move
  together the most?"

**Rules and linking constraints**
* "Write one more linking rule for my Pyomo model: *if I hold NVDA, I must also hold AMD*. Show it as math with
  $y_k$, and as one line of code using `model.y`."
* "Turn this business rule into a constraint: *at most two stocks from any one sector*. Explain which variables it
  uses and why."
* "My notebook says the rules cannot all be met. Here are my stocks and my `rules` and `extra_rules`. Which rule
  makes it impossible, and what is the smallest change that fixes it?"

**Reading the results**
* "Here is my rules ladder table. Some rules cost nothing. Explain what a non-binding constraint is, using my
  numbers."
* "Here is my backtest scorecard. Why is the promised average month higher than the average month, and what does
  the gap tell me?"
* "My fund beat *X*% of the monkeys over nine months. Is that skill or luck? How could I test it more fairly?"

**Your leaderboard entry**
* "Help me write a one-line theme and a one-sentence reason for my fund, based on my stocks and rules. Plain words,
  no hype."

If Claude's answer disagrees with your notebook, trust the notebook first, and ask Claude why the two differ.

### Your turn

Answer in a text cell:

1. Which stocks did you download, and did any fail or get dropped? (Part 1)
2. Which stock had the best average month, and how much of the money did the model put in it **without** rules?
3. What did the rules cost at the Step 6 risk limit, in percentage points a month and in dollars on \$10,000?
4. Change one rule (for example `max_share` to 0.20, or `max_stocks` to 12) and re-run Steps 5 and 6. What changed?
5. Look at Part 3. Where does your orange line go flat, and which rule is holding it back? Try `max_share = 0.20` and see what happens to the flat point.
6. Look at Part 4. In 2026, the months the model never saw, did your rules beat no rules? Did either beat the S&P 500? How far was each model's promise from what it earned?

## Where this is going: the final project

Everything tonight ran once, by hand, on five years of history. In the final project the same code will run **on its
own every month in the cloud**: download the newest prices, re-solve your model with your rules, and record what your
orange-line portfolio would have held, so you can check how it actually did on months the model has not seen yet. Dr. Dave will show you how
that is set up; the notebook you finished tonight is the engine.

# Bonus · Why does the orange line go flat? Compare each side of the constraints

Every constraint has a **left-hand side** (what the portfolio actually comes to) and a **right-hand side** (the limit).
The gap between them is the **slack**.

- **Slack = 0:** the constraint is **binding**. It is what holds the answer back. Loosen it, and the return can go up.
- **Slack > 0:** the constraint has room to spare. Loosening it changes nothing.

Below we check every constraint at **the top-left corner** from Part 3 (change `chosen_limit` to check any spot you
like), and then far **past** the flat point, for comparison.

In [ ]:
def check_constraints(name, limit):
    result = solve_portfolio(limit, use_rules=True)
    print(name, "- risk limit %.6f, best return %.2f%% a month" % (limit, 100 * result['return']))

    held = []
    for s in stocks:
        if result['weights'][s] > 0.001:
            held.append(s)

    # the main constraints: left-hand side vs right-hand side
    total = 0
    for s in stocks:
        total += result['weights'][s]
    rows = {}
    rows['invest all the money (sum of x = 1)'] = [total, 1]
    rows['keep risk under the limit (variance <= limit)'] = [result['risk'], limit]
    for sector in rules['must_have_sectors']:
        count = 0
        for s in held:
            if my_stocks.get(s) == sector:
                count += 1
        rows['at least one %s stock (number held >= 1)' % sector] = [count, 1]
    if rules['max_stocks'] is not None:
        rows['at most %d stocks (number held <= %d)' % (rules['max_stocks'], rules['max_stocks'])] = [len(held), rules['max_stocks']]

    table = pd.DataFrame(rows, index=['left-hand side', 'right-hand side']).T
    table['slack'] = (table['right-hand side'] - table['left-hand side']).abs()
    table['binding?'] = 'no'
    table.loc[table['slack'] < 1e-6, 'binding?'] = 'YES'
    display(table.round(6))

    # the linking constraints, one pair per stock held: 2% floor <= x <= 10% cap
    stock_rows = {}
    for s in held:
        x = result['weights'][s]
        where = 'in between'
        if rules['max_share'] is not None and x >= rules['max_share'] - 0.001:
            where = 'AT the cap (binding)'
        if rules['min_share'] is not None and x <= rules['min_share'] + 0.001:
            where = 'AT the floor (binding)'
        stock_rows[s] = [round(x, 4), where]
    display(pd.DataFrame(stock_rows, index=['share x', 'where it sits']).T.sort_values('share x', ascending=False))
    print()


# the spot to analyze: the top-left corner from Part 3, unless you type your own risk limit
chosen_limit = corner_risk
# chosen_limit = 0.0015

check_constraints("YOUR CHOSEN SPOT (the top-left corner unless you changed it)", chosen_limit)
check_constraints("FAR PAST the flat point, for comparison", top)

**What to notice.** On the climbing part of the orange line, the risk constraint is binding (slack 0): allow a little
more risk and the return goes up. Far past the flat point, the risk constraint has
lots of slack, and the stocks are stacked at exactly 10%: now the **cap** is the binding constraint. That is why the
line goes flat. Raising the risk limit loosens a constraint that was not holding anything back. To move the flat line
up, you would have to loosen the cap itself (try `max_share = 0.20` in Step 5).